In [0]:
# =============================================================
# CELDA 1: SETUP — Credenciales, Imports, Config, Batch ID
# =============================================================
import boto3
import json
import uuid
from datetime import datetime, timezone
from pyspark.sql.functions import current_timestamp, lit

# --- 1. Batch ID único por ejecución (trazabilidad MLOps) ---
BATCH_ID = str(uuid.uuid4())
BATCH_TIMESTAMP = datetime.now(timezone.utc).isoformat()
print(f"🔑 Batch ID: {BATCH_ID}")
print(f"🕐 Timestamp: {BATCH_TIMESTAMP}")

# --- 2. Cargar Credenciales ---
# Prioridad: Databricks secrets (producción) → archivo local (desarrollo)
try:
    config = {
        "aws_access_key": dbutils.secrets.get(scope="aws", key="access_key"),
        "aws_secret_key": dbutils.secrets.get(scope="aws", key="secret_key"),
    }
    print("✅ Credenciales cargadas desde Databricks Secrets.")
except Exception:
    try:
        with open("aws_secrets.json", "r") as f:
            config = json.load(f)
        print("✅ Credenciales cargadas desde aws_secrets.json (local).")
    except FileNotFoundError:
        raise SystemExit("❌ Credenciales no disponibles. Configura Databricks Secrets (scope='aws', keys='access_key','secret_key') o coloca aws_secrets.json.")
    except json.JSONDecodeError:
        raise SystemExit("❌ aws_secrets.json tiene formato inválido.")

# --- 3. Cliente Boto3 ---
REGION = "us-east-1"
BUCKET = "bronce-scrap-date"

s3_client = boto3.client(
    "s3",
    aws_access_key_id=config["aws_access_key"],
    aws_secret_access_key=config["aws_secret_key"],
    region_name=REGION,
)

# --- 4. Config Spark S3 (reutilizable) ---
S3_OPTIONS = {
    "fs.s3a.access.key": config["aws_access_key"],
    "fs.s3a.secret.key": config["aws_secret_key"],
    "fs.s3a.endpoint": "s3.amazonaws.com",
}

print(f"✅ Bucket: {BUCKET} | Region: {REGION}")

In [ ]:
# =============================================================
# CELDA 2: FUNCIONES — Descubrimiento + Procesamiento Idempotente
# =============================================================

def obtener_fuentes_disponibles(bucket):
    """Escanea carpetas en raw/ usando Delimiter='/' de S3."""
    print("📡 Escaneando carpetas en 'raw/'...")
    response = s3_client.list_objects_v2(Bucket=bucket, Prefix="raw/", Delimiter="/")

    fuentes = []
    if "CommonPrefixes" in response:
        for prefix in response["CommonPrefixes"]:
            nombre = prefix["Prefix"].replace("raw/", "").strip("/")
            if nombre:
                fuentes.append(nombre)

    print(f"✅ {len(fuentes)} fuentes encontradas: {fuentes}")
    return fuentes


def listar_parquets(bucket, fuente):
    """Lista TODOS los .parquet en raw/{fuente}/batches/ con paginación."""
    prefix = f"raw/{fuente}/batches/"
    paginator = s3_client.get_paginator("list_objects_v2")
    archivos = []

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        if "Contents" in page:
            for obj in page["Contents"]:
                if obj["Key"].endswith(".parquet"):
                    archivos.append(obj["Key"])

    return archivos


def archivar_archivo(bucket, key_origen, fuente):
    """Mueve un archivo de batches/ a archive/ (copy + delete)."""
    nombre_archivo = key_origen.split("/")[-1]
    key_destino = f"raw/{fuente}/archive/{nombre_archivo}"

    s3_client.copy_object(
        Bucket=bucket,
        CopySource={"Bucket": bucket, "Key": key_origen},
        Key=key_destino,
    )
    s3_client.delete_object(Bucket=bucket, Key=key_origen)
    return key_destino


def procesar_fuente(fuente, bucket):
    """
    Procesa todos los parquets de una fuente con idempotencia:
    1. Lee cada parquet con Spark
    2. Enriquece con metadata MLOps
    3. Escribe a Delta Bronze (append + mergeSchema)
    4. Mueve archivo a archive/ SOLO si write exitoso
    """
    archivos = listar_parquets(bucket, fuente)

    if not archivos:
        print(f"  ⏩ {fuente}: Sin archivos .parquet en batches/")
        return {"ok": 0, "error": 0, "archivados": 0}

    print(f"\n🚀 {fuente.upper()}: {len(archivos)} archivo(s) a procesar")

    contadores = {"ok": 0, "error": 0, "archivados": 0}
    ruta_bronze = f"s3a://{bucket}/bronze/{fuente}/"

    for key in archivos:
        nombre_archivo = key.split("/")[-1]
        ruta_s3a = f"s3a://{bucket}/{key}"

        # --- PASO 1: Leer parquet ---
        try:
            df_raw = spark.read.format("parquet")
            for k, v in S3_OPTIONS.items():
                df_raw = df_raw.option(k, v)
            df_raw = df_raw.load(ruta_s3a)
        except Exception as e:
            print(f"  ❌ ERROR lectura [{nombre_archivo}]: {e}")
            contadores["error"] += 1
            continue

        # --- PASO 2: Enriquecer con metadata ---
        df_enriched = (
            df_raw
            .withColumn("ingestion_timestamp", current_timestamp())
            .withColumn("source_file", lit(nombre_archivo))
            .withColumn("batch_id", lit(BATCH_ID))
            .withColumn("source_system", lit(fuente))
        )

        # --- PASO 3: Escribir a Delta Bronze ---
        try:
            writer = df_enriched.write.format("delta").mode("append").option("mergeSchema", "true")
            for k, v in S3_OPTIONS.items():
                writer = writer.option(k, v)
            writer.save(ruta_bronze)

            contadores["ok"] += 1
            print(f"  ✅ Guardado: {nombre_archivo} ({df_raw.count()} filas)")
        except Exception as e:
            print(f"  ❌ ERROR escritura [{nombre_archivo}]: {e}")
            contadores["error"] += 1
            continue  # NO mover → reintento en próxima ejecución

        # --- PASO 4: Archivar (SOLO si write exitoso) ---
        try:
            destino = archivar_archivo(bucket, key, fuente)
            contadores["archivados"] += 1
        except Exception as e:
            print(f"  ⚠️ WARN guardado en Bronze pero no archivado [{nombre_archivo}]: {e}")

    print(f"  📊 {fuente}: {contadores['ok']} OK | {contadores['error']} error | {contadores['archivados']} archivados")
    return contadores

In [ ]:
# =============================================================
# CELDA 3: EJECUCIÓN — Orquestación + Resumen Global
# =============================================================

fuentes_detectadas = obtener_fuentes_disponibles(BUCKET)

resumen_global = {"total_ok": 0, "total_error": 0, "total_archivados": 0}

for fuente in fuentes_detectadas:
    resultado = procesar_fuente(fuente, BUCKET)
    resumen_global["total_ok"] += resultado["ok"]
    resumen_global["total_error"] += resultado["error"]
    resumen_global["total_archivados"] += resultado["archivados"]

# --- Resumen Final ---
print("\n" + "=" * 60)
print(f"🏁 RESUMEN BATCH {BATCH_ID[:8]}...")
print(f"   Fuentes procesadas: {len(fuentes_detectadas)}")
print(f"   Archivos guardados en Bronze: {resumen_global['total_ok']}")
print(f"   Archivos archivados (movidos): {resumen_global['total_archivados']}")
print(f"   Archivos con error (quedan en batches/): {resumen_global['total_error']}")
print("=" * 60)